
## Statistical Built-in Functions:

**Statistical built-in function**s are used to `analyze numerical data, measure relationships between columns, identify frequent values, and summarize datasets`.


## Important Statistical Functions:

| Function | Interview Definition | Real-Time Use |
|---|---|---|
| `corr()` | Calculates the correlation between two numerical columns. | Analyze the relationship between advertising spend and sales. |
| `covar_pop()` | Calculates population covariance between two columns. | Analyze how two business metrics vary together across the entire population. |
| `covar_samp()` | Calculates sample covariance between two columns. | Analyze how two metrics vary together in a sample. |
| `sample()` | Selects a random fraction of rows from a DataFrame. | Select a sample of customer records for testing. |
| `freqItems()` | Finds frequent items in specified columns using an approximate algorithm. | Identify frequently occurring products or cities. |
| `summary()` | Generates descriptive statistics for DataFrame columns. | Inspect count, mean, standard deviation, minimum, and maximum. |


## Create a sample dataset:

In [0]:
from pyspark.sql import functions as F

stats_df = spark.createDataFrame([
    ("Chennai", 10000.0, 100),
    ("Chennai", 20000.0, 200),
    ("Chennai", 30000.0, 300),
    ("Madurai", 15000.0, 150),
    ("Madurai", 25000.0, 250),
    ("Madurai", 35000.0, 350)
], ["Branch", "Revenue", "UnitsSold"])

stats_df.show()

## A. corr():

**corr()** is used to `calculates the Pearson correlation coefficient between two numeric columns, measuring the strength and direction of their linear relationship`.


## Real-time example:

**An e-commerce company wants to determine whether selling more units is associated with higher revenue.**

In [0]:
stats_df.stat.corr("Revenue", "UnitsSold")


## B. cov():

**cov()** is used to `calculates the sample covariance between two numeric columns, indicating whether their values tend to move together or in opposite directions`.

Unlike correlation, covariance is not restricted to a `range of -1 to +1`.

In [0]:
stats_df.stat.cov("Revenue", "UnitsSold")


## C. sample():

**sample()** is used to `returns a random sample of rows from a DataFrame using a specified sampling fraction`.

In [0]:
sample_df = stats_df.sample(
    withReplacement=False,
    fraction=0.5,
    seed=42
)

sample_df.show()


Here:
- `withReplacement=False:` A row cannot be selected multiple times through replacement.
- `fraction=0.5:` Each row has a 50% sampling probability.
- `seed=42:` Provides a repeatable random seed for reproducibility under the same execution conditions.

Important: A fraction of 0.5 does not guarantee that exactly 50% of rows will be returned. Sampling is probabilistic.



Stratified sampling with `sampleBy()`

**Suppose you want to sample rows separately within each branch.**

In [0]:
branch_sample = stats_df.sampleBy(
    "Branch",
    fractions={
        "Chennai": 0.5,
        "Madurai": 0.5
    },
    seed=42
)

branch_sample.show()


## D. freqItems():

**freqItems()** is used to `identifies frequent items in one or more columns` using a **probabilistic algorithm.**


In [0]:
category_df = spark.createDataFrame([
    ("Electronics",),
    ("Electronics",),
    ("Electronics",),
    ("Fashion",),
    ("Fashion",),
    ("Grocery",),
    ("Grocery",),
    ("Grocery",)
], ["Category"])

category_df.stat.freqItems(
    ["Category"],
    support=0.3
).show(truncate=False)


Here, `support=0.3` specifies a support `threshold of 30%`.


- **Electronics:** 3 out of 8 rows = 37.5%
- **Fashion:** 2 out of 8 rows = 25%
- **Grocery:** 3 out of 8 rows = 37.5%


## Important limitation:

**freqItems()** uses a `probabilistic algorith`m. 
Its output can include `false positives`, so it should not be treated as an exact frequency count.

**If you need exact category counts, use:**

In [0]:
category_df.groupBy("Category").count().show()

## E. summary():

summary() is used to `generates descriptive statistics for a DataFrame`, including **counts, means, standard deviations, minimums, and maximums** for applicable columns.


## Real-time example:

**Before building a sales dashboard, a data engineer wants to understand the distribution of revenue and units sold.**

In [0]:
stats_df.summary().show()

In [0]:
stats_df.select( "Branch", "Revenue",  "UnitsSold").summary(
    "count", "min", "max", "mean", "stddev"
).show()